## 06 - XDec Deconvolution (complete pipeline: Stage 0 -> k-sweep -> Stage 1 -> Stage 2)

### Aim
* Run XDec (`BRL-BCM/XDec_SM`) end-to-end on the full 182-sample TCGA-PAAD cohort, using `Zenodo 3969339` (Peng et al. 2019, Besca-reprocessed) as the Stage-0 reference - the decision locked in notebook 04's comparison table (also carried into this notebook below), chosen for full whole-tumor coverage, PDAC specificity, and comparability to the PMID 36274954 benchmark paper.
* **Kernel note:** this notebook runs on the **Python 3** kernel, not this project's usual `ir` (R) kernel - a deliberate, explicit exception (see notebook 04's resolved open decision). XDec itself is a Python CLI tool; running it natively here avoids the R-shells-out-to-Python indirection used in notebooks 05/05b.

### Input
* `data/processed/TCGA-PAAD_counts_raw.tsv` (raw counts, notebook 02) - XDec's bulk input.
* `data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad` - Peng et al.'s reprocessed single-cell reference (57,423 cells, 35 patients).

### Output
* `data/deconvolution/xdec/refs.tsv` / `ref_metadata.tsv` - the Peng-derived reference, exported with the `expm1()` fix (see the comparison table's "Technical wrinkle" note below).
* `data/deconvolution/xdec/stage0/selected_probes.txt` - the real, whole-tumor one-vs-rest informative gene set (344 genes).
* `data/deconvolution/xdec/full_sweep/k_XX/stability_scores.tsv` per k (10 partitions, one per individual k value), plus the combined `stability_scores.tsv`/`optimal_k.txt` and `timing_log.tsv` - the full `k=3..12` stability sweep, 3 of the 10 partitions running concurrently at any time, so an unattended, lid-closed run surviving hours doesn't lose everything to a single interruption.
* `data/deconvolution/xdec/stage1/proportions.tsv`, `cell_type_profiles.tsv`, `correlation.tsv` - Stage 1 output at the chosen k.
* `data/deconvolution/xdec/stage2/cell_type_profiles.tsv` - Stage 2's per-cell-type expression profiles via NNLS.

### Steps
1. Export Peng et al.'s reference to `--refs`/`--ref-metadata` TSVs (inlined natively here - handles the z-scored/natural-log wrinkle).
2. Run `xdec.py stage0 --skip-stability` for fast, real gene selection only (one-vs-rest t-test against all 10 `celltype1` classes).
3. Run the full `k=3..12` stability sweep as **10 partitions, one per individual k value** (each its own `xdec.py stage0 --k-range <k>` subprocess, checkpointed, skip-if-done), **3 partitions running concurrently** at any time via a thread pool - partition count (10, fixed by the k-range) and concurrency level (3, a separate hardware-driven choice) are independent. See step 3's markdown cells for why partitioned, why 3 concurrent, and the documented RNG side effect this has vs. a single combined call.
4. Stage 1 at the chosen (highest-stability) k.
5. Stage 2 - per-cell-type expression profiles.

* Note: **The algorithm parameters (statistical tests, convergence criteria, randomness, the number of subsets/repetitions in the stability test) is entirely XDec's out-of-the-box default** — none of it has been tuned. What we've changed is purely about **execution flow**: whether to additionally run the stability sweep within a single call, and whether to split the k-sweep across multiple calls instead of one.

### Result
* Steps 1-2 executed previously (real export: 17,004 genes x 2,748 cells; real Stage-0 selection: 344 genes, ~9 sec) under the prior R-kernel version of this notebook - re-executing here under Python reproduces the same result (same underlying `xdec.py`, same inputs). Step 3 is running (3-concurrent sweep, in progress as of this writing). Steps 4-5 pending until step 3 finishes.

### Conclusion
* Complete, self-contained, native-Python record of the XDec pipeline (reference choice -> export -> gene selection -> k-sweep -> Stage 1 -> Stage 2), matching notebook 04's planned pipeline steps 3-5. Every step is idempotent and resumable (skips/continues from whatever's already on disk). 
* The pipeline still needs refinement to make it more structured and reproducible, with flexible settings for parallel execution and analysis, using a workflow management tool such as Snakemake or SoS.

## Comparing candidate scRNA-seq references for XDec Stage 0

(Carried over from notebook 04's planning comparison, so this notebook is self-contained.) Every PDAC single-cell/reference source checked, GEO-restricted and not. "Verified" = checked directly (file listing, download, or object inspection), not taken from a secondary summary.

| Reference | Resource | Cell type coverage | Sample size | Note |
|---|---|---|---|---|
| `GSE111672` (Moncada 2020) | GEO | General (ductal/cancer/stromal/immune); one generic `Fibroblasts` label, no subtype split | 2 patients, 3,659 cells | **Verified on download**: only 5 fibroblast cells in PDAC-A, 0 in PDAC-B - no CAF-subtype resolution possible |
| `GSE155698` (Steele 2020) | GEO | Immune-focused (T/myeloid/NKT); epithelial/stromal secondary | 17 tumor + 3 normal | Risks Stage-0's t-test being dominated by immune-vs-rest contrasts; sorting status not independently confirmed |
| `GSE205049`/`GSE205354` (Hwang 2023) | GEO | Immune-dysfunction focus; scRNA+spatial+CyTOF | Treatment-naive PDAC + normal + blood | Sorting markers unverified (paywalled primary source) |
| `GSE154778` | GEO | Whole dissociated tissue - not FACS-sorted, no ready labels, full coverage by construction | 10 primary + 6 metastatic (16 patients) | Would need its own clustering/annotation sub-pipeline first - self-derived labels, same caveat as proxy clusters |
| `GSE129455` (Elyada 2019) | GEO | Fibroblast-enriched (sorted CD45−/EPCAM−/CD31−); defines myCAF/iCAF/apCAF | 4 tumors | **Mouse only** - human data from the same study is dbGaP, not GEO |
| `GSE212966`, `GSE263733` | GEO | Unverified | Unverified | Title-only hits, not independently checked |
| Loveless et al. 2024 (`Zenodo 14199536`) | Zenodo | Named **CXCL10+ fibroblast** subtype; broad atlas (~21 cell types); validated directly against TCGA-PAAD bulk | 229 patients, ~1M cells | **Verified via Zenodo API**: single 33.4 GB file - exceeds 16GB RAM, infeasible |
| `researchdata.se/2025-64` | SND | Named **myCAF/iCAF/ifCAF** | 5 patients | **Restricted access** - requires a formal data-access request |
| HTAN | Synapse | Large, rigorously curated, multi-consortium | 200-257 patients, 1.6M+ cells | Synapse account + DUA required; even larger than Loveless et al. - same RAM infeasibility |
| CELLxGENE Census | Census | Unknown | - | XDec has native tooling for this (`census_adapter.py`) but no PDAC CAF-annotated entry confirmed |
| `CAF_integrated.rds`, considered then superseded | Zenodo `6024273` (integrates `GSE111672`+`GSE154778`+`GSE155698`+`GSM4293555`+Peng et al.+OUGS) | **CAF/fibroblast-only** - all 10,504 cells are fibroblasts; 7 unlabeled `seurat_clusters` used as a proxy for subtype | 10,504 cells | **Ran successfully** (262 real genes, no errors) - but the one-vs-rest test it produced is CAF-internal only ("cluster X vs other CAF clusters"), not a whole-tumor contrast. Superseded once a full-coverage PDAC-specific option was confirmed accessible. |
| Peng et al. 2019 / `CRA001160`, raw | GSA | ~10 cell types, PDAC-specific | 35 patients (24 PDAC + 11 normal) | **Verified on the GSA entry**: 70 files, 2.69 TB raw sequencing runs only - infeasible to reprocess from scratch |
| **Peng et al. 2019, reprocessed - chosen** | **Zenodo `3969339`** ("Peng2019 PRJCA001063 ... for Besca") | **Full coverage, PDAC-specific, better stromal split than any GEO-only option**: ductal (21,487), **fibroblast (6,926) + pancreatic stellate cell (5,862) as two separate stromal classes**, endothelial (9,027), acinar, endocrine, detailed immune (macrophage, B cell, CD4/CD8 T-cell subsets incl. regulatory/proliferating, myeloid DC, plasma cells) | **57,423 cells**, 35 patients - the real, (near-)complete Peng et al. dataset, reprocessed, not a subset | **Downloaded and verified** (`StdWf1_PRJCA001063_annotated.h5ad`, 1,681,192,568 bytes, exact match); clean HGNC gene symbols. **Technical wrinkle** (see note below) |

**Technical wrinkle found on inspection of `Zenodo 3969339` - not a blocker, but must be handled correctly:** neither matrix in the annotated h5ad is literal raw integer counts.
* `adata.X` (the main matrix) is z-scored/scaled - contains **negative values** (range -4.18 to 10.0) - not usable directly as `--refs` (XDec expects non-negative expression).
* `adata.raw.X` is non-negative (0 to 8.93) but is **natural-log-normalized** (`ln(x+1)` via `sc.pp.log1p()`), not `log2(x+1)`. XDec's `--log2-refs` flag inverts `log2(x+1)` specifically (`2^x - 1`) - applying it here would use the wrong base and produce incorrect values.
* **Fix (inlined below):** back-transform `adata.raw.X` with `expm1()` (= `e^x - 1`, the correct inverse for natural log) before writing the `--refs` TSV, then run XDec **without** `--log2-refs` (the data is no longer log-transformed by that point).
* Separately noticed: this file's cell barcodes (e.g. `T1_AAACCTGAGATGTCGG`) match the barcode format seen in `CAF_integrated.rds`'s `Patient2` column (`CA001063_T1`) - confirming Peng et al.'s cells are indeed one of the five source datasets integrated into `CAF_integrated.rds`.

**Why `Zenodo 3969339` (Peng et al., reprocessed) wins over everything else:** it is the only option that is simultaneously (a) full whole-tumor cell-type coverage, (b) PDAC-specific rather than a different/smaller pancreatic study, (c) large (57K cells, 35 patients), (d) the *exact* source data behind the PMID 36274954 benchmark paper, and (e) small enough to load on this laptop (1.68 GB). Every other option failed at least one of these.

### Set working directory to the project root

In [1]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "rnaseq_download":
    os.chdir(os.path.join("..", ".."))
os.getcwd()

'/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress'

### Step 1: export Peng et al.'s reference (with the expm1 fix), inlined natively

In [2]:
xdec_repo = "tools/XDec_SM"
xdec_script = os.path.join(xdec_repo, "python", "xdec.py")
if not os.path.exists(xdec_script):
    os.makedirs(os.path.dirname(xdec_repo) or ".", exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/BRL-BCM/XDec_SM.git", xdec_repo], check=True)
assert os.path.exists(xdec_script)

xdec_dir = "data/deconvolution/xdec"
os.makedirs(xdec_dir, exist_ok=True)
refs_path = os.path.join(xdec_dir, "refs.tsv")
meta_path = os.path.join(xdec_dir, "ref_metadata.tsv")

if not os.path.exists(refs_path):
    import scanpy as sc

    adata = sc.read_h5ad("data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad")
    class_col = "celltype1"
    max_cells_per_class = 300
    rng = np.random.default_rng(1)

    classes = adata.obs[class_col].astype(str)
    keep_names = []
    for cls, names in classes.groupby(classes).groups.items():
        names = np.array(names)  # cell barcodes (obs_names), not positional indices
        n = min(max_cells_per_class, len(names))
        keep_names.extend(rng.choice(names, size=n, replace=False))

    sub_raw = adata.raw[keep_names]
    counts_like = np.expm1(sub_raw.X)
    try:
        counts_like = counts_like.toarray()
    except AttributeError:
        pass
    counts_like = np.asarray(counts_like)
    counts_like[counts_like < 0] = 0.0  # guard against floating-point noise below zero

    cells = pd.Index(keep_names)
    refs_df = pd.DataFrame(counts_like.T, index=sub_raw.var_names, columns=cells)
    refs_df.index.name = "Genes"
    refs_df.to_csv(refs_path, sep="\t")

    meta_df = pd.DataFrame({"Sample_ID": cells, "Cell_Type": classes.loc[cells].values})
    meta_df.to_csv(meta_path, sep="\t", index=False)

    print(f"Wrote {refs_df.shape[0]} genes x {refs_df.shape[1]} cells -> {refs_path}")
    print(meta_df["Cell_Type"].value_counts().to_string())
else:
    print("refs.tsv already exported - skipping.")

refs.tsv already exported - skipping.


### Step 2: Stage 0 gene selection (`--skip-stability` - fast, probe selection only)

In [3]:
import time

stage0_dir = os.path.join(xdec_dir, "stage0")
os.makedirs(stage0_dir, exist_ok=True)
probes_path = os.path.join(stage0_dir, "selected_probes.txt")

if not os.path.exists(probes_path):
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage0",
         "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--refs", refs_path,
         "--ref-metadata", meta_path,
         "--skip-stability",
         "--outdir", stage0_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print(f"Stage 0 (gene selection only) elapsed: {time.time() - t0:.1f} sec")

with open(probes_path) as fh:
    probes = [line.strip() for line in fh if line.strip()]
len(probes)

344

### Step 3: full `k=3..12` stability sweep - 10 partitions, one per individual k value, 3 running concurrently at any time

To be explicit about two separate facts that "partitioned by k, run 3 at a time in parallel" can blur together: (a) **the partition granularity** - there are 10 partitions total, one per individual k value (k=3, k=4, ..., k=12 - each its own subprocess, not k divided into a few chunks/ranges), and (b) **the concurrency level** - 3 of those 10 partitions run at once, the rest queue behind them. These are independent choices: the partition count (10) is fixed by the k-range itself, while the concurrency (3) is a separate hardware-driven choice (see the design note below) and could be set to 1 (fully sequential), 10 (fully parallel), or anything in between without changing what a "partition" means.

**Why partitioned at all:** this run can take hours. `cmd_stage0()` only writes `stability_scores.tsv`/`optimal_k.txt` once, after the entire `k_range` loop finishes - no incremental save inside a single combined call. Any non-clean interruption before the full range completes means zero usable output. Partitioning into one subprocess per k, each writing its own `k_XX/stability_scores.tsv` immediately, means only the currently running k's are at risk.

**Why 3 concurrent workers:** this machine is an Apple M3, 8 cores (4 performance + 4 efficiency), 16GB RAM. Each k's subprocess uses ~130% CPU (~1.3 cores) based on earlier observation - 3 concurrent workers (~4 cores) leaves headroom for the OS/Jupyter/everything else during an hours-long unattended run, rather than saturating all 8 cores and risking thermal throttling on a sustained multi-hour load.

**Documented side effect of partitioning (unchanged by concurrency level - verified from source):** `estimate_stability()` creates `rng = np.random.default_rng(seed)` once per call, before its k loop. Every single-k partition (whether run sequentially or concurrently with others) resets to the same `--seed` (default `12345`) and starts consuming the random stream from the beginning - so every partition draws the same first segment of random sample-subsets, rather than the segment it would get at its true position in one combined `--k-range 3,4,...,12` call. Stability scores from this sweep will be close to, but not bit-identical with, a single combined run. How many partitions run concurrently (1, 3, or 10) does not change this effect at all - it's a property of partitioning by individual k, not of execution order or concurrency.

#### Execution Time Evaluation & Resource Management Strategy

**Previously measured execution time (`CAF_integrated.rds`, 262-gene list, for reference):**

| k | Elapsed | Stability score |
|---|---|---|
| 3 | 616.4 sec (~10.3 min) | 0.9992 |
| 4 | 1167.6 sec (~19.5 min) | 0.9941 |
| 5 | 2015.0 sec (~33.6 min) | 1.0000 |
| 6 | killed ~32-33 min in, incomplete | - |

**Real-time measured execution for this run (`Zenodo 3969339` Peng et al., 344-gene list, 3 concurrent workers):**

| k | Elapsed | Stability score |
|---|---|---|
| 3 | 971.4 sec (~16.2 min) | 1.0000 |
| 4 | 1877.9 sec (~31.3 min) | 0.9239 |
| 5 | 2871.7 sec (~47.9 min) | 0.9680 |

**Gene-count effect, isolated by comparing the same k across both references:** 344 genes (Peng) vs. 262 genes (CAF) is a +31% increase in gene count, and the real slowdown is 1.58x at k=3, 1.61x at k=4, 1.43x at k=5 - a fairly consistent ~1.4-1.6x multiplier. This is real but qualitatively different from k's effect: it does not compound as k grows (the ratio stays in a narrow band across k=3,4,5), consistent with gene count adding roughly linear per-solve cost (longer data vectors in each QP/NNLS fit), as opposed to k's worse-than-linear cost (more unknowns per solve) that was already shown to compound sharply (CAF k=3->4->5 ratios: 1.89x, 1.73x; Peng k=3->4->5 ratios: 1.93x, 1.53x - both references show the same steep, k-driven growth pattern, independent of which reference was used).

**Rationale for the partition + 3-concurrent-worker strategy, grounded in this data:**
* **Partitioning (10 separate k-calls, not 1 combined):** justified independent of timing - `cmd_stage0()` only saves results once, after the whole `k_range` loop finishes, so an interruption during an hours-long unattended run would otherwise lose everything. Both tables above confirm this isn't a brief run: even the fastest k (k=3) takes 10-16 minutes, and cost keeps climbing - an interruption at, say, k=9 on a single combined call would discard all of k=3-8's completed work along with it.
* **3 concurrent workers, not more:** this machine has 8 cores (4P+4E per `sysctl`), and a single k's subprocess measured at ~100-130% CPU (~1-1.3 cores) when run alone. 3 workers (~3-4 cores) leaves headroom for the OS/Jupyter during a multi-hour unattended run rather than saturating all 8 cores. The real parallel numbers above (k=3,4,5 run concurrently) are higher than the sequential CAF numbers partly because of real gene-count difference (344 vs 262, ~1.4-1.6x alone) and partly likely some contention from running 3 at once - the two effects aren't cleanly separated in this data, since no sequential Peng-reference baseline was measured before switching to parallel.
* **Why not more workers:** given cost already climbs steeply with k alone, adding contention on top (by running more than 3-4 concurrently) risks each individual k slowing down enough to offset the concurrency benefit, on top of the thermal-throttling risk already flagged for a sustained multi-hour, all-cores-saturated load.

In [4]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

sweep_dir = os.path.join(xdec_dir, "full_sweep")
os.makedirs(sweep_dir, exist_ok=True)
timing_log_path = os.path.join(sweep_dir, "timing_log.tsv")
timing_log_lock = threading.Lock()
if not os.path.exists(timing_log_path):
    with open(timing_log_path, "w") as fh:
        fh.write("k\telapsed_sec\tstability_score\n")


def run_one_k(k):
    k_dir = os.path.join(sweep_dir, f"k_{k:02d}")
    k_stab_tsv = os.path.join(k_dir, "stability_scores.tsv")

    if os.path.exists(k_stab_tsv):
        print(f"k={k} already done - skipping.")
        return

    os.makedirs(k_dir, exist_ok=True)
    print(f"Running k={k} ...")
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage0",
         "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--probes", probes_path,
         "--k-range", str(k),
         "--outdir", k_dir],
        capture_output=True, text=True,
    )
    elapsed_k = time.time() - t0

    if not os.path.exists(k_stab_tsv):
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"k={k} did not produce stability_scores.tsv - stopping here "
            "so completed k's are not lost. See log above."
        )

    stab_k = pd.read_csv(k_stab_tsv, sep="\t")
    score_k = stab_k.loc[stab_k["k"] == k, "stability_score"].iloc[0]

    with timing_log_lock:
        with open(timing_log_path, "a") as fh:
            fh.write(f"{k}\t{elapsed_k:.1f}\t{score_k:.4f}\n")
    print(f"k={k} done: {elapsed_k:.1f} sec, stability={score_k:.4f}")


# 3 workers in parallel - see the markdown cell above for why 3 on this hardware.
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(run_one_k, k): k for k in range(3, 13)}
    for future in as_completed(futures):
        future.result()  # re-raises if run_one_k raised, surfacing the error here

pd.read_csv(timing_log_path, sep="\t")

k=3 already done - skipping.
k=4 already done - skipping.
k=5 already done - skipping.
k=6 already done - skipping.
k=7 already done - skipping.
k=8 already done - skipping.
k=9 already done - skipping.
k=10 already done - skipping.
k=12 already done - skipping.
k=11 already done - skipping.


,k,elapsed_sec,stability_score
0,3,971.4,1.0000
1,4,1877.9,0.9239
2,5,2871.7,0.9680
3,6,5068.6,0.9580
4,9,7544.9,0.9909
5,7,7587.5,0.9507
6,8,11484.8,0.9995
7,10,13793.9,0.9870
8,11,16240.9,0.8848
9,12,14954.4,0.8774


### Aggregate all completed k's into the combined stability result

In [5]:
import glob

k_dirs = sorted(glob.glob(os.path.join(sweep_dir, "k_*", "stability_scores.tsv")))
assert len(k_dirs) > 0

all_stab = pd.concat([pd.read_csv(p, sep="\t") for p in k_dirs], ignore_index=True)
all_stab = all_stab.sort_values("k").reset_index(drop=True)

all_stab.to_csv(os.path.join(sweep_dir, "stability_scores.tsv"), sep="\t", index=False)

optimal_k = int(all_stab.loc[all_stab["stability_score"].idxmax(), "k"])
with open(os.path.join(sweep_dir, "optimal_k.txt"), "w") as fh:
    fh.write(str(optimal_k) + "\n")

print(f"Recommended k = {optimal_k} (stability = {all_stab['stability_score'].max():.4f})")
all_stab

Recommended k = 3 (stability = 1.0000)


,k,stability_score
0,3,1.000000
1,4,0.923927
2,5,0.968030
3,6,0.958004
4,7,0.950703
5,8,0.999548
6,9,0.990895
7,10,0.986968
8,11,0.884765
9,12,0.877396


### Step 4: Stage 1 - deconvolve at the chosen k

In [6]:
stage1_dir = os.path.join(xdec_dir, "stage1")
os.makedirs(stage1_dir, exist_ok=True)
proportions_path = os.path.join(stage1_dir, "proportions.tsv")

if not os.path.exists(proportions_path):
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage1",
         "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--probes", probes_path,
         "--k", str(optimal_k),
         "--outdir", stage1_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print(f"Stage 1 elapsed: {time.time() - t0:.1f} sec")

proportions = pd.read_csv(proportions_path, sep="\t", index_col=0)
proportions.head()

,Profile_1,Profile_2,Profile_3
TCGA-IB-7888-01A-11R-2156-07,0.096816,0.623379,0.279805
TCGA-FB-A78T-01A-12R-A32O-07,0.445064,0.403589,0.151347
TCGA-2J-AABI-01A-12R-A41B-07,0.724221,0.082459,0.193320
TCGA-HZ-7918-01A-11R-2156-07,0.099318,0.557294,0.343387
TCGA-2J-AABO-01A-21R-A41B-07,0.096230,0.810188,0.093581


### Step 5: Stage 2 - recover cell-type-specific expression profiles (NNLS, one-shot - not partitioned, per notebook 04's reasoning)

In [7]:
stage2_dir = os.path.join(xdec_dir, "stage2")
os.makedirs(stage2_dir, exist_ok=True)
# Stage 2 actually writes stage2_means.tsv / stage2_stderrs.tsv, not
# cell_type_profiles.tsv (that name is Stage 1's output) - confirmed by
# inspecting the real output after Stage 2 ran; the original guess here was wrong.
means_path = os.path.join(stage2_dir, "stage2_means.tsv")
stderrs_path = os.path.join(stage2_dir, "stage2_stderrs.tsv")

if not os.path.exists(means_path):
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage2",
         "--bulk-counts", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--proportions", proportions_path,
         "--outdir", stage2_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print(f"Stage 2 elapsed: {time.time() - t0:.1f} sec")

stage2_means = pd.read_csv(means_path, sep="\t", index_col=0)
stage2_stderrs = pd.read_csv(stderrs_path, sep="\t", index_col=0)
stage2_means.head()

,Profile_1,Profile_2,Profile_3
5S_rRNA,0.477308,0.313488,2.626024
5_8S_rRNA,0.000000,5.417520,0.174918
7SK,0.248836,0.217802,0.000000
A1BG,0.000000,6.171198,30.763223
A1BG-AS1,0.000000,18.284811,140.096641
